In [2]:
import cv2
import os
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from skimage.feature import local_binary_pattern

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [3]:
def extract_lbp(face_gray):
    face = cv2.resize(face_gray, (128, 128))
    blurred = cv2.GaussianBlur(face, (5, 5), 1.0)

    P = 8
    R = 1
    METHOD = "uniform"

    lbp = local_binary_pattern(blurred, P, R, METHOD)

    n_bins = P + 2
    hist, _ = np.histogram(
        lbp.ravel(),
        bins=n_bins,
        range=(0, n_bins)
    )

    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)

    return hist


In [4]:
def extract_laplacian(face_gray):
    face = cv2.resize(face_gray, (128, 128))
    blurred = cv2.GaussianBlur(face, (3, 3), 0)

    edges = cv2.Canny(face, 50, 150)
    lap = cv2.Laplacian(blurred, cv2.CV_64F)
    abs_lap = np.abs(lap)

    masked = abs_lap[edges > 0]

    if len(masked) == 0:
        return np.zeros(3)

    mean = masked.mean()
    std  = masked.std()
    var  = masked.var()

    return np.array([mean, std, var])


In [5]:
def process_data(face_gray):
    lbp_feat = extract_lbp(face_gray)
    lap_feat = extract_laplacian(face_gray)

    feature = np.concatenate([lbp_feat, lap_feat])

    return feature


In [6]:
def load_data(folder):
    X, y = [], []

    color_dir = os.path.join(folder, "color")

    for file in os.listdir(color_dir):
        if not file.endswith(".jpg"):
            continue

        label = 1 if "real" in file.lower() else 0

        img_path = os.path.join(color_dir, file)
        img = cv2.imread(img_path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(
            gray, scaleFactor=1.3, minNeighbors=5
        )

        if len(faces) == 0:
            continue

        x, y0, w, h = faces[0]
        face = gray[y0:y0+h, x:x+w]

        feature = process_data(face)

        X.append(feature)
        y.append(label)

    return np.array(X), np.array(y)

In [7]:
def train_model(X_train, y_train, model_path="rf_lbp_lap_ver2.pkl"):
    clf = RandomForestClassifier(
        n_estimators=600,
        max_depth=40,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train, y_train)

    joblib.dump(clf, model_path)
    print(f"Model saved to: {model_path}")

    return clf

In [8]:
train_folder = "data/train_img/train_img"
test_folder  = "data/test_img/test_img"

print("Loading train data...")
X_train, y_train = load_data(train_folder)

print("Loading test data...")
X_test, y_test = load_data(test_folder)

print("Training model...")
model = train_model(X_train, y_train)

print("Evaluating...")
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Loading train data...
Loading test data...
Training model...
Model saved to: rf_lbp_lap_ver2.pkl
Evaluating...
Accuracy: 0.8020541549953315
              precision    recall  f1-score   support

           0       0.80      0.97      0.88      1601
           1       0.79      0.29      0.43       541

    accuracy                           0.80      2142
   macro avg       0.80      0.63      0.65      2142
weighted avg       0.80      0.80      0.77      2142



In [9]:
import cv2
import numpy as np
import joblib
from skimage.feature import local_binary_pattern


class FaceAntiSpoofing:
    def __init__(self, model_path, cam_id=0, fake_threshold=0.90):
        self.model = joblib.load(model_path)
        print("Model loaded:", model_path)

        self.fake_threshold = fake_threshold

        self.face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
        )

        self.cap = cv2.VideoCapture(cam_id)

        if not self.cap.isOpened():
            raise RuntimeError("Cannot open camera")

    def _extract_lbp(self, face_gray):
        face = cv2.resize(face_gray, (128, 128))
        blurred = cv2.GaussianBlur(face, (5, 5), 1.0)

        P, R = 8, 1
        lbp = local_binary_pattern(blurred, P, R, method="uniform")

        n_bins = P + 2
        hist, _ = np.histogram(
            lbp.ravel(),
            bins=n_bins,
            range=(0, n_bins)
        )

        hist = hist.astype("float")
        hist /= (hist.sum() + 1e-6)

        return hist

    def _extract_laplacian(self, face_gray):
        face = cv2.resize(face_gray, (128, 128))
        blurred = cv2.GaussianBlur(face, (3, 3), 0)

        edges = cv2.Canny(face, 50, 150)
        lap = cv2.Laplacian(blurred, cv2.CV_64F)
        abs_lap = np.abs(lap)

        masked = abs_lap[edges > 0]

        if len(masked) == 0:
            return np.zeros(3)

        return np.array([
            masked.mean(),
            masked.std(),
            masked.var()
        ])


    def _extract_feature(self, face_gray):
        lbp_feat = self._extract_lbp(face_gray)
        lap_feat = self._extract_laplacian(face_gray)
        return np.concatenate([lbp_feat, lap_feat])

    def _predict(self, face_gray):
        feature = self._extract_feature(face_gray).reshape(1, -1)
        pred = self.model.predict(feature)[0]
        prob = self.model.predict_proba(feature)[0]
        return pred, np.max(prob)

    def run(self):
        print("Press 'q' to quit")

        while True:
            ret, frame = self.cap.read()
            if not ret:
                break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            faces = self.face_cascade.detectMultiScale(
                gray,
                scaleFactor=1.3,
                minNeighbors=5
            )

            for (x, y, w, h) in faces:
                face_gray = gray[y:y+h, x:x+w]

                pred, conf = self._predict(face_gray)

                # label = "REAL" if pred == 1 else "FAKE"
                # color = (0, 255, 0) if pred == 1 else (0, 0, 255)
                if pred == 0 and conf >= self.fake_threshold:
                    label = f"FAKE ({conf:.2f})"
                    color = (0, 0, 255)
                else:
                    label = f"REAL ({conf:.2f})"
                    color = (0, 255, 0)

                cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)
                cv2.putText(
                    frame,
                    f"{label} ({conf:.2f})",
                    (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    color,
                    2
                )

            cv2.imshow("Face Anti-Spoofing", frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        self.cap.release()
        cv2.destroyAllWindows()

In [ ]:

fas = FaceAntiSpoofing(model_path="rf_lbp_lap.pkl", fake_threshold=0.8)
fas.run()

Model loaded: rf_lbp_lap.pkl
Press 'q' to quit


: 